# Notebook 02 — Mapper Analysis

**Goal:** Build a Mapper graph from integrated multi-omics data to visualize the topological structure of the sample space. Identify nodes enriched in resilient vs accelerated individuals.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_utils import generate_synthetic_multimics, preprocess_omics, integrate_multiomics
from mapper_utils import build_mapper_graph, auto_mapper, enrich_mapper_nodes, compare_mapper_graphs, export_mapper_html
from config import RANDOM_SEED

%matplotlib inline
print('✅ Imports OK')

## 1. Generate & Integrate Multi-Omics Data

In [ ]:
ds = generate_synthetic_multimics(n_samples=300, topology_type='circle', noise=0.06, n_features=40)

# Preprocess each layer
omics_dict = {
    'transcriptomics': preprocess_omics(pd.DataFrame(ds['transcriptomics']), method='standard'),
    'metabolomics': preprocess_omics(pd.DataFrame(ds['metabolomics']), method='standard'),
    'epigenomics': preprocess_omics(pd.DataFrame(ds['epigenomics']), method='standard'),
}

# Integrate
integrated = integrate_multiomics(omics_dict)
labels = ds['labels']

print(f"Integrated data shape: {integrated.shape}")
print(f"Groups: {labels.value_counts().to_dict()}")

## 2. Build Mapper Graph (Accelerated vs Resilient)

In [ ]:
# Full dataset Mapper
graph_full = auto_mapper(integrated, labels=labels, verbose=True)

# Per-group Mapper
accel_idx = (labels == 'accelerated').values
resil_idx = (labels == 'resilient').values

graph_accel = auto_mapper(integrated[accel_idx], verbose=False)
graph_resil = auto_mapper(integrated[resil_idx], verbose=False)

print('✅ All Mapper graphs built')

## 3. Structural Comparison

In [ ]:
comparison = compare_mapper_graphs(graph_accel, graph_resil)
print(f"{'Metric':<25} {'Accelerated':>15} {'Resilient':>15}")
print('-' * 55)
for metric in ['n_nodes', 'n_edges', 'connected_components', 'mean_degree']:
    print(f"{metric:<25} {comparison['graph_a'][metric]:>15} {comparison['graph_b'][metric]:>15}")

## 4. Node Enrichment Analysis

Which Mapper nodes are enriched in resilient individuals?

In [ ]:
enrichment = enrich_mapper_nodes(graph_full, labels, group_col='group')

print(f"Total nodes: {len(enrichment)}")
print(f"\nTop nodes by size:")
display(enrichment.sort_values('size', ascending=False).head(10))

# Nodes dominated by resilient
resilient_nodes = enrichment[enrichment['dominant_group'] == 'resilient']
print(f"\nResilient-dominated nodes: {len(resilient_nodes)}/{len(enrichment)} "
      f"({100*len(resilient_nodes)/len(enrichment):.1f}%)")

## 5. Export Interactive HTML

In [ ]:
# Export full Mapper graph as interactive HTML
try:
    export_mapper_html(graph_full, path='../results/figures/mapper_full.html')
    print('✅ Mapper graph exported to results/figures/mapper_full.html')
except Exception as e:
    print(f'⚠️  HTML export failed (may need kmapper installed): {e}')

## 6. Key Findings

- The Mapper graph reveals [XX] connected components
- Resilient individuals cluster in [YY] nodes
- **Biological interpretation:** resilient individuals share a common topological neighborhood, suggesting conserved multi-omics architecture